# OCR Receipt - Inference Results Visualization

This notebook runs end-to-end OCR inference and visualizes the results with bounding boxes and recognized text.

In [ ]:
import sys
from pathlib import Path

import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ultralytics import YOLO

# Add src to path
sys.path.insert(0, str(Path('.').resolve()))
from src.infer import load_recognition_checkpoint, recognize_text
from src.config import get_project_paths

print("✅ Imports successful")

## Configuration

In [ ]:
# Paths
paths = get_project_paths()
image_path = "data/vn_receipt/images/train/mcocr_public_145013aagqw.jpg"  # Change this to your image
detector_path = "artifacts/detector_runs/yolo_textdet/weights/best.pt"
recognizer_path = "artifacts/checkpoints/recognition_best.pt"

# Verify files exist
assert Path(image_path).exists(), f"Image not found: {image_path}"
assert Path(detector_path).exists(), f"Detector not found: {detector_path}"
assert Path(recognizer_path).exists(), f"Recognizer not found: {recognizer_path}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")
print(f"✅ Image: {image_path}")
print(f"✅ Detector: {detector_path}")
print(f"✅ Recognizer: {recognizer_path}")

## Load Models

In [ ]:
print("Loading YOLO detector...")
yolo = YOLO(detector_path)
print("✅ YOLO loaded")

print("Loading recognition model...")
model, tokenizer = load_recognition_checkpoint(Path(recognizer_path), device)
print(f"✅ Recognition model loaded")
print(f"   Vocab size: {tokenizer.vocab_size}")

## Run Inference

In [ ]:
print(f"Running inference on: {image_path}")
results = recognize_text(
    image_path,
    yolo,
    model.encoder,
    model.decoder,
    tokenizer,
    device,
    max_len=100
)

print(f"✅ Found {len(results)} text regions")
print("\nDetected texts:")
for i, item in enumerate(results[:10], 1):  # Show first 10
    print(f"  {i}. {item['text'][:50]} (conf: {item['confidence']:.2f})")
if len(results) > 10:
    print(f"  ... and {len(results) - 10} more")

## Visualize Results

In [ ]:
# Load image
image = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
height, width = image_rgb.shape[:2]

# Create figure
fig, ax = plt.subplots(1, 1, figsize=(16, 12))
ax.imshow(image_rgb)

# Draw bounding boxes and text
colors = plt.cm.tab20(np.linspace(0, 1, len(results)))

for i, item in enumerate(results):
    box = item['box']
    text = item['text']
    confidence = item['confidence']
    
    x1, y1, x2, y2 = box
    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
    
    # Draw rectangle
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=2,
        edgecolor=colors[i % len(colors)],
        facecolor='none',
        alpha=0.7
    )
    ax.add_patch(rect)
    
    # Add text label
    label = f"{text} ({confidence:.2f})"
    ax.text(
        x1, y1 - 5,
        label,
        fontsize=9,
        color=colors[i % len(colors)],
        bbox=dict(facecolor='white', alpha=0.7, pad=2),
        verticalalignment='bottom'
    )

ax.set_title(f"OCR Results - {len(results)} text regions detected", fontsize=16, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('ocr_results_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Visualization saved as 'ocr_results_visualization.png'")

## Summary

In [ ]:
# Print all results in a readable format
print("\n" + "="*80)
print("FULL OCR RESULTS")
print("="*80)
print(f"Image: {image_path}")
print(f"Total detections: {len(results)}")
print(f"Average confidence: {np.mean([r['confidence'] for r in results]):.4f}")
print("\nDetailed Results:")
print("-" * 80)

for i, item in enumerate(results, 1):
    box = [int(x) for x in item['box']]
    text = item['text']
    conf = item['confidence']
    print(f"{i:3d}. [{box[0]:4d}, {box[1]:4d}, {box[2]:4d}, {box[3]:4d}] | Conf: {conf:.4f} | Text: {text}")

print("="*80)

## Export Results as JSON

In [ ]:
import json

# Save results to JSON
output_json = {
    'image': image_path,
    'num_detections': len(results),
    'results': results
}

with open('ocr_results.json', 'w', encoding='utf-8') as f:
    json.dump(output_json, f, indent=2, ensure_ascii=False)

print("✅ Results saved to 'ocr_results.json'")